<a href="https://colab.research.google.com/github/Poojarautela03/ABTALKS/blob/main/day-18-hallucination-detection/hallucination-detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q google-generativeai faiss-cpu numpy

import time
import json
import numpy as np
import faiss
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
chat_model = genai.GenerativeModel('gemini-3.6-flash')
print("Setup done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 86.2 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Setup done.


In [3]:
questions = [
    # History (5)
    {"q": "In what year did World War II end?", "truth": "1945", "domain": "history"},
    {"q": "Who was the first President of the United States?", "truth": "George Washington", "domain": "history"},
    {"q": "In what year did India gain independence?", "truth": "1947", "domain": "history"},
    {"q": "Who wrote the Indian national anthem?", "truth": "Rabindranath Tagore", "domain": "history"},
    {"q": "What year did the Berlin Wall fall?", "truth": "1989", "domain": "history"},

    # Science (5)
    {"q": "What is the chemical symbol for gold?", "truth": "Au", "domain": "science"},
    {"q": "How many planets are in our solar system?", "truth": "8", "domain": "science"},
    {"q": "What is the speed of light in a vacuum (approx)?", "truth": "300,000 km/s (approximately 299,792 km/s)", "domain": "science"},
    {"q": "What gas do plants absorb during photosynthesis?", "truth": "Carbon dioxide", "domain": "science"},
    {"q": "What is the boiling point of water at sea level in Celsius?", "truth": "100°C", "domain": "science"},

    # Geography (5)
    {"q": "What is the capital of Australia?", "truth": "Canberra", "domain": "geography"},
    {"q": "What is the longest river in the world?", "truth": "The Nile (or Amazon, depending on measurement method)", "domain": "geography"},
    {"q": "Which is the largest desert in the world?", "truth": "Antarctic Desert (or Sahara if considering hot deserts only)", "domain": "geography"},
    {"q": "What is the smallest country in the world by area?", "truth": "Vatican City", "domain": "geography"},
    {"q": "Which mountain is the tallest in the world?", "truth": "Mount Everest", "domain": "geography"},

    # Technology (5) — deliberately includes fictional/obscure to test fabrication
    {"q": "Who is the founder of Nimbus Robotics?", "truth": "This is a fictional company — no real founder exists", "domain": "technology"},
    {"q": "What year was Python programming language first released?", "truth": "1991", "domain": "technology"},
    {"q": "What does 'HTTP' stand for?", "truth": "HyperText Transfer Protocol", "domain": "technology"},
    {"q": "Who founded the company 'Zylotech Dynamics'?", "truth": "This is a fictional company — no real founder exists", "domain": "technology"},
    {"q": "What year was the first iPhone released?", "truth": "2007", "domain": "technology"}
]

print(f"Total questions: {len(questions)}")

Total questions: 20


In [4]:
no_context_results = []

for i, item in enumerate(questions):
    print(f"[{i+1}/20] {item['q']}")
    prompt = f"Answer this question directly and concisely: {item['q']}"

    try:
        response = chat_model.generate_content(prompt)
        answer = response.text.strip()
    except Exception as e:
        answer = f"[ERROR: {e}]"

    no_context_results.append({
        "question": item['q'],
        "ground_truth": item['truth'],
        "domain": item['domain'],
        "response": answer
    })
    time.sleep(13)  # stay under free-tier rate limit

print("\nAll 20 no-context queries done.")

[1/20] In what year did World War II end?
[2/20] Who was the first President of the United States?
[3/20] In what year did India gain independence?
[4/20] Who wrote the Indian national anthem?
[5/20] What year did the Berlin Wall fall?
[6/20] What is the chemical symbol for gold?
[7/20] How many planets are in our solar system?


[8/20] What is the speed of light in a vacuum (approx)?


[9/20] What gas do plants absorb during photosynthesis?


[10/20] What is the boiling point of water at sea level in Celsius?


[11/20] What is the capital of Australia?


...

[17/20] What year was Python programming language first released?


[18/20] What does 'HTTP' stand for?


[19/20] Who founded the company 'Zylotech Dynamics'?


[20/20] What year was the first iPhone released?



All 20 no-context queries done.


In [5]:
# Small KB combining Nimbus/Zylotech facts + general reference snippets
# so RAG has *something* to retrieve for both real and fictional questions
knowledge_base = [
    "Nimbus Robotics is a fictional company used for testing purposes; no real-world founder or history exists for it.",
    "Zylotech Dynamics is a fictional company used for testing purposes; no real-world founder or history exists for it.",
    "World War II ended in 1945.",
    "George Washington was the first President of the United States, serving from 1789 to 1797.",
    "India gained independence from British rule on August 15, 1947.",
    "Rabindranath Tagore wrote 'Jana Gana Mana', India's national anthem.",
    "The Berlin Wall fell in November 1989.",
    "Gold's chemical symbol is Au, from the Latin 'aurum'.",
    "The solar system has 8 planets, following the reclassification of Pluto as a dwarf planet in 2006.",
    "The speed of light in a vacuum is approximately 299,792 kilometers per second.",
    "Plants absorb carbon dioxide during photosynthesis and release oxygen.",
    "Water boils at 100 degrees Celsius at standard sea-level atmospheric pressure.",
    "Canberra is the capital city of Australia.",
    "The Nile River is traditionally considered the longest river in the world, though some studies argue the Amazon is longer.",
    "The Antarctic Desert is the largest desert in the world by area; the Sahara is the largest hot desert.",
    "Vatican City is the smallest country in the world by both area and population.",
    "Mount Everest is the tallest mountain above sea level, at 8,849 meters.",
    "Python programming language was first released in 1991 by Guido van Rossum.",
    "HTTP stands for HyperText Transfer Protocol.",
    "The first iPhone was released by Apple in 2007."
]

def get_embeddings(texts, model="models/gemini-embedding-001"):
    embeddings = []
    for text in texts:
        result = genai.embed_content(model=model, content=text)
        embeddings.append(result['embedding'])
        time.sleep(1)
    return embeddings

kb_embeddings = get_embeddings(knowledge_base)
kb_vectors = np.array(kb_embeddings, dtype=np.float32)
dimension = kb_vectors.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(kb_vectors)
print(f"FAISS index size: {index.ntotal}")

def retrieve_chunks(query, top_k=3):
    query_embedding = get_embeddings([query])[0]
    query_vector = np.array([query_embedding], dtype=np.float32)
    distances, indices = index.search(query_vector, top_k)
    return [(knowledge_base[idx], float(dist)) for idx, dist in zip(indices[0], distances[0])]

FAISS index size: 20


In [6]:
rag_results = []

for i, item in enumerate(questions):
    print(f"[{i+1}/20] {item['q']}")
    retrieved = retrieve_chunks(item['q'], top_k=3)
    context = "\n".join(f"- {chunk}" for chunk, dist in retrieved)

    prompt = f"""Answer using ONLY the context below. If the context doesn't answer the question, say so.

CONTEXT:
{context}

QUESTION:
{item['q']}

ANSWER:"""

    try:
        response = chat_model.generate_content(prompt)
        answer = response.text.strip()
    except Exception as e:
        answer = f"[ERROR: {e}]"

    rag_results.append({
        "question": item['q'],
        "ground_truth": item['truth'],
        "domain": item['domain'],
        "retrieved_chunks": retrieved,
        "response": answer
    })
    time.sleep(13)

print("\nAll 20 RAG queries done.")

[1/20] In what year did World War II end?


[2/20] Who was the first President of the United States?


[3/20] In what year did India gain independence?


[4/20] Who wrote the Indian national anthem?


[5/20] What year did the Berlin Wall fall?


...

[17/20] What year was Python programming language first released?


[18/20] What does 'HTTP' stand for?


[19/20] Who founded the company 'Zylotech Dynamics'?


[20/20] What year was the first iPhone released?



All 20 RAG queries done.


In [7]:
# Fill in after manually classifying both sets, same criteria as Cell 4
no_context_hallucinated_count = 0  # e.g. 6 out of 20
rag_hallucinated_count = 0          # e.g. 1 out of 20

no_context_rate = (no_context_hallucinated_count / 20) * 100
rag_rate = (rag_hallucinated_count / 20) * 100

print(f"No-context hallucination rate: {no_context_rate:.1f}%")
print(f"RAG hallucination rate: {rag_rate:.1f}%")
print(f"Reduction: {no_context_rate - rag_rate:.1f} percentage points")

No-context hallucination rate: 0.0%
RAG hallucination rate: 0.0%
Reduction: 0.0 percentage points


In [8]:
import re

def signal_1_unsupported_claims(response, context):
    """Flags numbers, dates, or capitalized named entities in the response not present in context."""
    numbers = re.findall(r'\b\d{3,4}\b', response)  # years, big numbers
    entities = re.findall(r'\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)*\b', response)  # naive named entity pattern

    unsupported = []
    for n in numbers:
        if n not in context:
            unsupported.append(n)
    for e in entities:
        if e not in context and e not in ["The", "This", "Answer", "Context"]:
            unsupported.append(e)

    flagged = len(unsupported) > 0
    return flagged, unsupported

def signal_2_jaccard_overlap(response, context, threshold=0.15):
    """Token-level Jaccard similarity between response and retrieved context."""
    resp_tokens = set(response.lower().split())
    ctx_tokens = set(context.lower().split())

    if not resp_tokens or not ctx_tokens:
        return True, 0.0

    intersection = len(resp_tokens & ctx_tokens)
    union = len(resp_tokens | ctx_tokens)
    jaccard = intersection / union if union > 0 else 0

    flagged = jaccard < threshold
    return flagged, jaccard

def signal_3_contradiction_check(response, context):
    """Secondary LLM call checking for contradiction."""
    prompt = f"""Context: {context}

Response: {response}

Does this response contradict the provided context? Answer only "yes" or "no"."""

    try:
        result = chat_model.generate_content(prompt)
        answer = result.text.strip().lower()
        flagged = "yes" in answer
        return flagged, answer
    except Exception as e:
        return False, f"[ERROR: {e}]"

def signal_4_citation_absence(response, context):
    """Flags if response makes factual claims but never references the context."""
    has_factual_claim = bool(re.search(r'\d|is|was|are|were', response))
    references_context = any(word.lower() in response.lower() for word in context.split()[:20])

    flagged = has_factual_claim and not references_context
    return flagged, references_context


def run_detector(question, response, retrieved_chunks):
    context = " ".join(chunk for chunk, dist in retrieved_chunks)

    s1_flag, s1_detail = signal_1_unsupported_claims(response, context)
    s2_flag, s2_score = signal_2_jaccard_overlap(response, context)
    s3_flag, s3_detail = signal_3_contradiction_check(response, context)
    time.sleep(13)  # rate limit for the secondary LLM call
    s4_flag, s4_detail = signal_4_citation_absence(response, context)

    flags_triggered = sum([s1_flag, s2_flag, s3_flag, s4_flag])
    confidence_score = 1 - (flags_triggered / 4)  # 1.0 = high confidence, 0.0 = low

    return {
        "question": question,
        "signal_1_unsupported_claims": {"flagged": s1_flag, "detail": s1_detail},
        "signal_2_jaccard_overlap": {"flagged": s2_flag, "score": round(s2_score, 4)},
        "signal_3_contradiction": {"flagged": s3_flag, "detail": s3_detail},
        "signal_4_citation_absence": {"flagged": s4_flag, "references_context": s4_detail},
        "flags_triggered": flags_triggered,
        "confidence_score": confidence_score
    }

In [9]:
detector_results = []

for r in rag_results:
    result = run_detector(r['question'], r['response'], r['retrieved_chunks'])
    detector_results.append(result)

    print(f"Q: {result['question']}")
    print(f"  Signal 1 (unsupported claims): {result['signal_1_unsupported_claims']['flagged']}")
    print(f"  Signal 2 (Jaccard overlap): {result['signal_2_jaccard_overlap']['score']} "
          f"({'FLAGGED' if result['signal_2_jaccard_overlap']['flagged'] else 'ok'})")
    print(f"  Signal 3 (contradiction): {result['signal_3_contradiction']['flagged']}")
    print(f"  Signal 4 (citation absence): {result['signal_4_citation_absence']['flagged']}")
    print(f"  Confidence score: {result['confidence_score']:.2f} ({result['flags_triggered']}/4 flags)\n")

with open('hallucination_detector_results.json', 'w') as f:
    json.dump(detector_results, f, indent=2)

Q: In what year did World War II end?
  Signal 1 (unsupported claims): True
  Signal 2 (Jaccard overlap): 0.0339 (FLAGGED)
  Signal 3 (contradiction): False
  Signal 4 (citation absence): False
  Confidence score: 0.50 (2/4 flags)



Q: Who was the first President of the United States?
  Signal 1 (unsupported claims): True
  Signal 2 (Jaccard overlap): 0.0462 (FLAGGED)
  Signal 3 (contradiction): False
  Signal 4 (citation absence): False
  Confidence score: 0.50 (2/4 flags)



Q: In what year did India gain independence?
  Signal 1 (unsupported claims): True
  Signal 2 (Jaccard overlap): 0.0323 (FLAGGED)
  Signal 3 (contradiction): False
  Signal 4 (citation absence): False
  Confidence score: 0.50 (2/4 flags)



Q: Who wrote the Indian national anthem?
  Signal 1 (unsupported claims): True
  Signal 2 (Jaccard overlap): 0.0323 (FLAGGED)
  Signal 3 (contradiction): False
  Signal 4 (citation absence): False
  Confidence score: 0.50 (2/4 flags)



Q: What year did the Berlin Wall fall?
  Signal 1 (unsupported claims): True
  Signal 2 (Jaccard overlap): 0.0175 (FLAGGED)
  Signal 3 (contradiction): False
  Signal 4 (citation absence): False
  Confidence score: 0.50 (2/4 flags)



...

Q: Who is the founder of Nimbus Robotics?
  Signal 1 (unsupported claims): True
  Signal 2 (Jaccard overlap): 0.0308 (FLAGGED)
  Signal 3 (contradiction): False
  Signal 4 (citation absence): False
  Confidence score: 0.50 (2/4 flags)



Q: What year was Python programming language first released?
  Signal 1 (unsupported claims): True
  Signal 2 (Jaccard overlap): 0.0323 (FLAGGED)
  Signal 3 (contradiction): False
  Signal 4 (citation absence): False
  Confidence score: 0.50 (2/4 flags)



Q: What does 'HTTP' stand for?
  Signal 1 (unsupported claims): True
  Signal 2 (Jaccard overlap): 0.0339 (FLAGGED)
  Signal 3 (contradiction): False
  Signal 4 (citation absence): False
  Confidence score: 0.50 (2/4 flags)



Q: Who founded the company 'Zylotech Dynamics'?
  Signal 1 (unsupported claims): True
  Signal 2 (Jaccard overlap): 0.0294 (FLAGGED)
  Signal 3 (contradiction): False
  Signal 4 (citation absence): False
  Confidence score: 0.50 (2/4 flags)



Q: What year was the first iPhone released?
  Signal 1 (unsupported claims): True
  Signal 2 (Jaccard overlap): 0.0312 (FLAGGED)
  Signal 3 (contradiction): False
  Signal 4 (citation absence): False
  Confidence score: 0.50 (2/4 flags)

